# Python Asynchronous Programming

---

## 1. What Are Asynchronous Tasks?

Asynchronous tasks are units of work that can be **paused and resumed** without blocking the execution of other tasks. Instead of waiting for a slow operation (like a network request or disk I/O), the program can move on and come back when the result is ready.

**Typical use cases:**
- Sending emails or notifications
- Fetching data from APIs or databases
- File I/O operations
- Recalculating/refreshing cached data in the background
- WebSocket connections or chat servers

```python
import asyncio

async def send_email(to: str):
    await asyncio.sleep(2)  # Simulates network delay
    print(f"Email sent to {to}")
```

---

## 2. Asynchronous vs Parallel Execution

| Feature | Asynchronous | Parallel |
|---|---|---|
| Threads/Processes | Single thread | Multiple threads/processes |
| Concurrency model | Cooperative (yield control) | Preemptive (OS-managed) |
| Best for | I/O-bound tasks | CPU-bound tasks |
| Overhead | Low | Higher |

- **Async**: One thread switches between tasks during waiting periods.
- **Parallel**: Multiple tasks literally run at the same time on different CPU cores.

```python
# Async: tasks interleave during I/O waits
async def task_a(): await asyncio.sleep(1)
async def task_b(): await asyncio.sleep(1)
# Both finish in ~1s total, not 2s
```

---

## 3. Event Loop

The **event loop** is the core of asyncio. It continuously monitors and dispatches events, deciding which coroutine to run next.

**How it works:**
1. Maintains a queue of ready-to-run callbacks and coroutines.
2. Runs a coroutine until it hits an `await`.
3. While waiting, picks up another ready task.
4. Returns to the suspended coroutine when its awaited result is ready.

```
[Task A] --await--> [Task B runs] --await--> [Task A resumes]
```

---

## 4. Async/Await Syntax

- `async def` defines a **coroutine function**.
- `await` suspends the coroutine until the awaited object completes.

```python
import asyncio

async def fetch_data():
    print("Fetching...")
    await asyncio.sleep(1)  # Non-blocking wait
    return {"data": 42}

async def main():
    result = await fetch_data()
    print(result)

asyncio.run(main())
```

---

## 5. Coroutines, Tasks, and Futures

### Coroutine
A function defined with `async def`. It does **not** run until awaited or scheduled.

```python
async def greet():
    return "Hello"

# greet() returns a coroutine object, not "Hello"
```

### Task
Wraps a coroutine and schedules it to run on the event loop **concurrently**.

```python
task = asyncio.create_task(greet())
result = await task
```

### Future
A low-level object representing a **result that hasn't been computed yet**. Tasks are a subclass of Future.

```python
loop = asyncio.get_event_loop()
future = loop.create_future()
future.set_result("done")
print(future.result())  # "done"
```

---

## 6. Getting the Event Loop

```python
import asyncio

# Recommended (Python 3.10+): use asyncio.run()
asyncio.run(main())

# Get the running loop from inside a coroutine
async def main():
    loop = asyncio.get_running_loop()

# Get or create a loop (legacy, outside async context)
loop = asyncio.get_event_loop()
```

---

## 7. Running and Stopping the Event Loop

```python
import asyncio

async def main():
    print("Running!")

# High-level: recommended approach
asyncio.run(main())

# Low-level (manual control)
loop = asyncio.new_event_loop()
try:
    loop.run_until_complete(main())
finally:
    loop.close()
```

To stop a loop programmatically:
```python
loop.stop()  # Stops after current iteration
```

---

## 8. Scheduling Callbacks — `call_soon`

`call_soon` schedules a **regular (non-coroutine) callback** to run in the next iteration of the event loop.

```python
import asyncio

def say_hello():
    print("Hello from callback!")

async def main():
    loop = asyncio.get_running_loop()
    loop.call_soon(say_hello)
    await asyncio.sleep(0)  # Yield control so callback runs

asyncio.run(main())
```

---

## 9. Scheduling Delayed Callbacks — `call_later`

`call_later` schedules a callback to run **after a delay** (in seconds).

```python
import asyncio

def reminder():
    print("Reminder triggered!")

async def main():
    loop = asyncio.get_running_loop()
    loop.call_later(2, reminder)   # Run after 2 seconds
    await asyncio.sleep(3)         # Keep loop alive

asyncio.run(main())
```

---

## 10. Libraries That Work with AsyncIO — AIOHTTP

**AIOHTTP** is an async HTTP client/server library built on top of asyncio.

```python
import aiohttp
import asyncio

async def fetch(url: str):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            return await response.text()

async def main():
    html = await fetch("https://example.com")
    print(html[:100])

asyncio.run(main())
```

Other notable async-compatible libraries:
- `asyncpg` — PostgreSQL async driver
- `motor` — Async MongoDB driver
- `aiobotocore` — AWS SDK async wrapper
- `FastAPI` — Async web framework

---

## 11. Chaining Coroutines — `gather`

`asyncio.gather()` runs multiple coroutines **concurrently** and waits for all to finish.

```python
import asyncio

async def task(name, delay):
    await asyncio.sleep(delay)
    return f"{name} done"

async def main():
    results = await asyncio.gather(
        task("A", 1),
        task("B", 2),
        task("C", 1),
    )
    print(results)  # All 3 finish in ~2s, not 4s

asyncio.run(main())
# ['A done', 'B done', 'C done']
```

---

## 12. Choosing Between Multiprocessing / Multithreading / Async

| Scenario | Best Choice |
|---|---|
| I/O-bound (network, disk) | `asyncio` |
| I/O-bound with blocking libs | `ThreadPoolExecutor` |
| CPU-bound (heavy computation) | `ProcessPoolExecutor` |

### Executor Objects

Executors allow running **blocking code** in a thread or process pool from within asyncio using `loop.run_in_executor()`.

```python
import asyncio
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def blocking_task(n):
    return n * n  # Simulate CPU work

async def main():
    loop = asyncio.get_running_loop()

    # Thread pool (I/O-bound blocking code)
    with ThreadPoolExecutor() as pool:
        result = await loop.run_in_executor(pool, blocking_task, 5)
        print("Thread result:", result)

    # Process pool (CPU-bound code)
    with ProcessPoolExecutor() as pool:
        result = await loop.run_in_executor(pool, blocking_task, 10)
        print("Process result:", result)

asyncio.run(main())
```

---

## 13. Using an Async Queue

`asyncio.Queue` is a thread-safe queue designed for **producer-consumer** patterns in async code.

```python
import asyncio

async def producer(queue: asyncio.Queue):
    for i in range(5):
        await queue.put(i)
        print(f"Produced: {i}")
        await asyncio.sleep(0.5)

async def consumer(queue: asyncio.Queue):
    while True:
        item = await queue.get()
        print(f"Consumed: {item}")
        queue.task_done()

async def main():
    queue = asyncio.Queue()
    await asyncio.gather(producer(queue), consumer(queue))

asyncio.run(main())
```

---

## 14. Async Generators and `async for`

**Async generators** use `async def` + `yield` to produce values asynchronously. They are consumed with `async for`.

```python
import asyncio

async def ticker(count: int):
    for i in range(count):
        await asyncio.sleep(0.5)
        yield i

async def main():
    async for value in ticker(5):
        print(f"Tick: {value}")

asyncio.run(main())
```

Useful for streaming data, paginated API responses, or reading large files chunk by chunk.

---

## 15. Custom Event Loop Implementations — uvloop

**uvloop** is a drop-in replacement for the default asyncio event loop, built on **libuv** (the same engine as Node.js). It can be **2–4x faster** than the default loop.

```bash
pip install uvloop
```

```python
import asyncio
import uvloop

async def main():
    await asyncio.sleep(1)
    print("Running on uvloop!")

# Method 1: set as default policy
asyncio.set_event_loop_policy(uvloop.EventLoopPolicy())
asyncio.run(main())

# Method 2: run directly
uvloop.run(main())
```

**When to use:**
- High-throughput servers (APIs, WebSockets)
- Applications processing thousands of concurrent connections
- Performance-critical async services

> ⚠️ uvloop is not available on Windows. Use it in Linux/macOS production environments.

---

## Summary Cheat Sheet

```
asyncio.run(main())              → Entry point
async def / await                → Define and call coroutines
asyncio.create_task()            → Schedule concurrent task
asyncio.gather()                 → Run multiple coroutines concurrently
asyncio.Queue()                  → Producer-consumer async queue
loop.call_soon()                 → Schedule immediate callback
loop.call_later(n, fn)           → Schedule delayed callback
loop.run_in_executor()           → Run blocking code in thread/process pool
async for / async def + yield    → Async iteration and generators
uvloop                           → High-performance event loop replacement
```